# 04 - Selección de Variables (Versión Corregida)

In [ ]:
# Librerías
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# ============================================================
# RUTAS ACTUALIZADAS
# ============================================================

PROJECT_ROOT = Path().resolve().parents[1]

PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
PATH_MODEL = PROJECT_ROOT / "data" / "model"

df_model = pd.read_csv(PATH_PROCESSED / "features_intra_anuales_2007-2024.csv")
clusters_mun = pd.read_csv(PATH_MODEL / "clusters_municipales.csv")

In [ ]:
# Merge clusters
df_model = df_model.merge(clusters_mun, on="municipio", how="left")
print(df_model["cluster"].value_counts())

In [ ]:
# Selección de variables
target = "Rendimiento (t/ha)"

corr = df_model.corr(numeric_only=True)[target].abs().sort_values(ascending=False)
vars_seleccion = corr[corr > 0.14].index.tolist()
vars_seleccion.remove(target)

X = df_model[vars_seleccion]
y = df_model[target]

In [ ]:
# Train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Lasso
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

coef = pd.Series(lasso.coef_, index=X.columns)
vars_lasso = coef[coef != 0].sort_values(key=abs, ascending=False)

print(vars_lasso)

In [ ]:
# Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

importancias = pd.Series(rf.feature_importances_, index=X.columns)
vars_rf_top = importancias.sort_values(ascending=False).head(20).index

print(importancias.sort_values(ascending=False).head(20))

In [ ]:
# Intersección final
vars_final = list(set(vars_lasso.index) & set(vars_rf_top))

print(f"Total variables seleccionadas: {len(vars_final)}")
print(vars_final)

In [ ]:
# ============================================================
# GUARDAR RESULTADOS
# ============================================================

ruta_vars = PATH_MODEL / "variables_seleccionadas.csv"
ruta_rf = PATH_MODEL / "importancia_variables_rf.csv"
ruta_lasso = PATH_MODEL / "coeficientes_lasso.csv"

pd.DataFrame({"variable": vars_final}).to_csv(ruta_vars, index=False)
importancias.sort_values(ascending=False).to_csv(ruta_rf)
vars_lasso.to_csv(ruta_lasso)

print("✔ Variables guardadas:", ruta_vars)
print("✔ Importancias RF guardadas:", ruta_rf)
print("✔ Coeficientes Lasso guardados:", ruta_lasso)